In [1]:
# %%
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.maskers import NiftiLabelsMasker
from nilearn import datasets, image

# === Define Subject IDs ===
subject_ids = [
    "sub-0001", "sub-0002", "sub-0003", "sub-0004", "sub-0005",
    "sub-0006", "sub-0007", "sub-0008", "sub-0009", "sub-0011"
]
# subject_ids = [
#     "sub-0001", "sub-0002"
# ]
# subject_ids = [
#     "sub-0001"
# ]

In [2]:
import nibabel as nib

# === Load preprocessed fMRI task data ===
img_task = nib.load("sub-0001_task-workingmemory_acq-seq_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz")
data_task = img_task.get_fdata()  # Shape: (X, Y, Z, T)
header_task = img_task.header
print("Header of fMRI task data:", header_task.get_zooms())
print("Shape of fMRI task data:", data_task.shape)

# === Load preprocessed fMRI resting-state data ===
img_rest = nib.load("sub-0001_task-restingstate_acq-seq_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz")
data_rest = img_rest.get_fdata()
header_rest = img_rest.header
print("Header of fMRI rest data:", header_rest.get_zooms())
print("Shape of fMRI rest data:", data_rest.shape)

Header of fMRI task data: (3.0, 3.0, 3.3, 2.0)
Shape of fMRI task data: (65, 77, 60, 160)
Header of fMRI rest data: (3.0, 3.0, 3.3, 2.0)
Shape of fMRI rest data: (65, 77, 60, 240)


In [3]:

fmri_rest_files = [
    f"{sid}_task-restingstate_acq-seq_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    for sid in subject_ids
]

fmri_task_files = [
    f"{sid}_task-workingmemory_acq-seq_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    for sid in subject_ids
]

tsv_files = [f"../{sid}.tsv" for sid in subject_ids]



In [4]:
# %%
# === Load Atlas and Define Masker ===
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100)
masker = NiftiLabelsMasker(atlas.maps, standardize=True)

[get_dataset_dir] Dataset found in C:\Users\elena\nilearn_data\schaefer_2018


In [5]:
# %%
import numpy as np
import pandas as pd

def build_trial_wise_matrix(time_series_task, time_series_rest, tsv_path, TR=2.0, fixed_len=3):
    """
    Επιστρέφει αντίστοιχα blocks task και rest για κάθε trial, με βάση τα onset του .tsv.

    Parameters:
    - time_series_task: NumPy array (T_task, R)
    - time_series_rest: NumPy array (T_rest, R)
    - tsv_path: string, path προς .tsv αρχείο
    - TR: χρονική διάρκεια TR σε δευτερόλεπτα
    - fixed_len: αριθμός TRs ανά trial block (default: 3)

    Returns:
    - task_matrix: NumPy array (n_trials, fixed_len, n_rois)
    - rest_matrix: NumPy array (n_trials, fixed_len, n_rois)
    """
    df = pd.read_csv(tsv_path, sep="\t")
    n_timepoints_task, n_rois = time_series_task.shape
    n_timepoints_rest = time_series_rest.shape[0]

    task_matrix = []
    rest_matrix = []

    for _, row in df.iterrows():
        onset = row["onset"]
        start_tr = int(np.floor(onset / TR))
        end_tr = start_tr + fixed_len

        # Skip trial αν δεν χωράει ούτε σε task ούτε σε rest
        if end_tr > n_timepoints_task or end_tr > n_timepoints_rest:
            continue

        block_task = time_series_task[start_tr:end_tr]
        block_rest = time_series_rest[start_tr:end_tr]

        task_matrix.append(block_task)
        rest_matrix.append(block_rest)

    return np.array(task_matrix), np.array(rest_matrix)

In [6]:
# %%
subject_task_data = {}
subject_rest_data = {}

for sid, rest_file, task_file, tsv_file in zip(subject_ids, fmri_rest_files, fmri_task_files, tsv_files):
    print(f"Processing {sid}")

    img_task = nib.load(task_file)
    img_rest = nib.load(rest_file)

    ts_task = masker.fit_transform(img_task)
    ts_rest = masker.transform(img_rest)

    task_matrix, rest_matrix = build_trial_wise_matrix(ts_task, ts_rest, tsv_file)

    subject_task_data[sid] = task_matrix
    subject_rest_data[sid] = rest_matrix

Processing sub-0001
Processing sub-0002
Processing sub-0003
Processing sub-0004
Processing sub-0005
Processing sub-0006
Processing sub-0007
Processing sub-0008
Processing sub-0009
Processing sub-0011


In [7]:
for sid,sid in zip(subject_task_data,subject_rest_data):
    print(f"Subject {sid} task data shape",subject_task_data[sid].shape)
    print(f"Subject {sid} rest data shape",subject_rest_data[sid].shape)

Subject sub-0001 task data shape (40, 3, 100)
Subject sub-0001 rest data shape (40, 3, 100)
Subject sub-0002 task data shape (40, 3, 100)
Subject sub-0002 rest data shape (40, 3, 100)
Subject sub-0003 task data shape (40, 3, 100)
Subject sub-0003 rest data shape (40, 3, 100)
Subject sub-0004 task data shape (40, 3, 100)
Subject sub-0004 rest data shape (40, 3, 100)
Subject sub-0005 task data shape (40, 3, 100)
Subject sub-0005 rest data shape (40, 3, 100)
Subject sub-0006 task data shape (40, 3, 100)
Subject sub-0006 rest data shape (40, 3, 100)
Subject sub-0007 task data shape (40, 3, 100)
Subject sub-0007 rest data shape (40, 3, 100)
Subject sub-0008 task data shape (40, 3, 100)
Subject sub-0008 rest data shape (40, 3, 100)
Subject sub-0009 task data shape (40, 3, 100)
Subject sub-0009 rest data shape (40, 3, 100)
Subject sub-0011 task data shape (40, 3, 100)
Subject sub-0011 rest data shape (40, 3, 100)


In [8]:
import numpy as np

def compute_roi_correlation_matrix(data):
    """
    Υπολογίζει Pearson correlation matrix μεταξύ ROIs,
    flattening όλα τα TRs από όλα τα trials.

    Parameters:
    - data: NumPy array (n_trials, 3, n_rois)

    Returns:
    - corr_matrix: NumPy array (n_rois, n_rois)
    """
    n_trials, n_trs, n_rois = data.shape
    flat_data = data.reshape(-1, n_rois)  # shape: (n_trials * 3, n_rois)

    corr_matrix = np.corrcoef(flat_data.T)  # (n_rois, n_rois)
    return corr_matrix


In [9]:
task_corr_dict = {}
rest_corr_dict = {}

for sid in subject_ids:
    task_corr = compute_roi_correlation_matrix(subject_task_data[sid])
    rest_corr = compute_roi_correlation_matrix(subject_rest_data[sid])

    task_corr_dict[sid]=task_corr
    rest_corr_dict[sid]=rest_corr


for sid,sid in zip(task_corr_dict,rest_corr_dict):
    print(f"Subject {sid} task data correlation",task_corr_dict[sid].shape)
    print(f"Subject {sid} rest data correlation",task_corr_dict[sid].shape)


Subject sub-0001 task data correlation (100, 100)
Subject sub-0001 rest data correlation (100, 100)
Subject sub-0002 task data correlation (100, 100)
Subject sub-0002 rest data correlation (100, 100)
Subject sub-0003 task data correlation (100, 100)
Subject sub-0003 rest data correlation (100, 100)
Subject sub-0004 task data correlation (100, 100)
Subject sub-0004 rest data correlation (100, 100)
Subject sub-0005 task data correlation (100, 100)
Subject sub-0005 rest data correlation (100, 100)
Subject sub-0006 task data correlation (100, 100)
Subject sub-0006 rest data correlation (100, 100)
Subject sub-0007 task data correlation (100, 100)
Subject sub-0007 rest data correlation (100, 100)
Subject sub-0008 task data correlation (100, 100)
Subject sub-0008 rest data correlation (100, 100)
Subject sub-0009 task data correlation (100, 100)
Subject sub-0009 rest data correlation (100, 100)
Subject sub-0011 task data correlation (100, 100)
Subject sub-0011 rest data correlation (100, 100)


In [50]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

def compute_partial_pearson_matrix(data, num_components=3):
    """
    Υπολογίζει πίνακα Partial Pearson Correlation μεταξύ ROIs,
    αφαιρώντας κοινή πληροφορία μέσω PCA (σε flatten δεδομένα από trials).

    Parameters:
    - data: NumPy array (n_trials, n_TRs, n_rois)
    - num_components: Αριθμός PCA components που θα αφαιρεθούν ως confounds

    Returns:
    - partial_corr_matrix: NumPy array (n_rois, n_rois) συμμετρικός
    """
    n_trials, n_trs, n_rois = data.shape
    flat_data = data.reshape(-1, n_rois)

    partial_corr_matrix = np.eye(n_rois)

    for i in range(n_rois):
        for j in range(i + 1, n_rois):
            X = flat_data[:, i].reshape(-1, 1)
            Y = flat_data[:, j].reshape(-1, 1)

            other_indices = [k for k in range(n_rois) if k != i and k != j]
            Z = flat_data[:, other_indices]

            num_components
            Z_pca = PCA(n_components=num_components).fit_transform(Z)

            def regress_out(A, Z_pca):
                reg = LinearRegression().fit(Z_pca, A)
                return A - reg.predict(Z_pca)

            X_resid = regress_out(X, Z_pca).flatten()
            Y_resid = regress_out(Y, Z_pca).flatten()

            corr = np.corrcoef(X_resid, Y_resid)[0, 1]
            partial_corr_matrix[i, j] = corr
            partial_corr_matrix[j, i] = corr

    return partial_corr_matrix

In [51]:
task_part_corr_dict = {}
rest_part_corr_dict = {}

for sid in subject_ids:
    task_part_corr = compute_partial_pearson_matrix(subject_task_data[sid])
    rest_part_corr = compute_partial_pearson_matrix(subject_rest_data[sid])

    task_part_corr_dict[sid]=task_part_corr
    rest_part_corr_dict[sid]=rest_part_corr
    print(f"Subject {sid} task data partial correlation",task_part_corr_dict[sid].shape)
    print(f"Subject {sid} rest data partial correlation",rest_part_corr_dict[sid].shape)


Subject sub-0001 task data partial correlation (100, 100)
Subject sub-0001 rest data partial correlation (100, 100)
Subject sub-0002 task data partial correlation (100, 100)
Subject sub-0002 rest data partial correlation (100, 100)
Subject sub-0003 task data partial correlation (100, 100)
Subject sub-0003 rest data partial correlation (100, 100)
Subject sub-0004 task data partial correlation (100, 100)
Subject sub-0004 rest data partial correlation (100, 100)
Subject sub-0005 task data partial correlation (100, 100)
Subject sub-0005 rest data partial correlation (100, 100)
Subject sub-0006 task data partial correlation (100, 100)
Subject sub-0006 rest data partial correlation (100, 100)
Subject sub-0007 task data partial correlation (100, 100)
Subject sub-0007 rest data partial correlation (100, 100)
Subject sub-0008 task data partial correlation (100, 100)
Subject sub-0008 rest data partial correlation (100, 100)
Subject sub-0009 task data partial correlation (100, 100)
Subject sub-00

In [10]:
import numpy as np
from sklearn.cross_decomposition import CCA

def compute_cca_matrix_across_trials(data, n_components=1):
    """
    Υπολογίζει Canonical Correlation μεταξύ κάθε ζεύγους ROIs,
    χωρίς flatten — χρησιμοποιώντας τα 3 TRs ανά trial ως μεταβλητές.

    Parameters:
    - data: NumPy array (n_trials, 3, n_rois)
    - n_components: αριθμός CCA components (συνήθως 1)

    Returns:
    - cca_matrix: NumPy array (n_rois, n_rois), συμμετρικός
    """
    n_trials, n_trs, n_rois = data.shape
    cca_matrix = np.eye(n_rois)

    for i in range(n_rois):
        for j in range(i + 1, n_rois):
            X = data[:, :, i]  # shape: (n_trials, 3)
            Y = data[:, :, j]  # shape: (n_trials, 3)

            cca = CCA(n_components=n_components)
            X_c, Y_c = cca.fit_transform(X, Y)

            # Υπολογίζουμε την Pearson correlation του 1ου canonical pair
            corr = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]
            cca_matrix[i, j] = corr
            cca_matrix[j, i] = corr

    return cca_matrix


In [11]:
task_cca_dict = {}
rest_cca_dict = {}

for sid in subject_ids:
    task_cca = compute_cca_matrix_across_trials(subject_task_data[sid])
    rest_cca = compute_cca_matrix_across_trials(subject_rest_data[sid])

    task_cca_dict[sid]=task_cca
    rest_cca_dict[sid]=rest_cca
    print(f"Subject {sid} task data cca",task_cca_dict[sid].shape)
    print(f"Subject {sid} rest data cca",task_cca_dict[sid].shape)

Subject sub-0001 task data cca (100, 100)
Subject sub-0001 rest data cca (100, 100)
Subject sub-0002 task data cca (100, 100)
Subject sub-0002 rest data cca (100, 100)


c:\Users\elena\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\cross_decomposition\_pls.py:113: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)


Subject sub-0003 task data cca (100, 100)
Subject sub-0003 rest data cca (100, 100)
Subject sub-0004 task data cca (100, 100)
Subject sub-0004 rest data cca (100, 100)
Subject sub-0005 task data cca (100, 100)
Subject sub-0005 rest data cca (100, 100)
Subject sub-0006 task data cca (100, 100)
Subject sub-0006 rest data cca (100, 100)
Subject sub-0007 task data cca (100, 100)
Subject sub-0007 rest data cca (100, 100)
Subject sub-0008 task data cca (100, 100)
Subject sub-0008 rest data cca (100, 100)
Subject sub-0009 task data cca (100, 100)
Subject sub-0009 rest data cca (100, 100)
Subject sub-0011 task data cca (100, 100)
Subject sub-0011 rest data cca (100, 100)


In [ ]:
from scipy.linalg import eig
import numpy as np

def compute_pcca_condition_wise_with_topkZ(data, i, j, cca_matrix, k=8, ridge=1e-3):
    """
    CONDITION-WISE PCCA (όχι trial-wise)
    Υπολογίζει την Partial CCA μεταξύ ROI i και ROI j, αφαιρώντας την επίδραση Ζ,
    με δείγματα = trials και features = 3 TRs/ROI.

    Parameters:
    - data: np.array shape (n_trials, 3, n_rois)
    - i, j: indices των ROIs για X και Y
    - cca_matrix: (n_rois, n_rois) condition-wise CCA matrix (για επιλογή Ζ)
    - k: πόσα ROIs στο Ζ (conditioning set)
    - ridge: μικρό ridge για σταθερότητα (στην παλινδρόμηση & αντιστροφές)

    Returns:
    - rho: πρώτη partial canonical correlation (scalar in [0,1])
    """
    # ===== Shapes / βασικά =====
    n_trials, n_trs, n_rois = data.shape
    assert n_trs == 3, "Περιμένω 3 TRs ανά trial."

    # ===== Helpers (inline, χωρίς extra συναρτήσεις) =====
    def zscore(M):
        M = M - M.mean(axis=0, keepdims=True)
        std = M.std(axis=0, ddof=1, keepdims=True)
        std[std == 0] = 1.0
        return M / std

    def cov(A, B):
        A0 = A - A.mean(axis=0, keepdims=True)
        B0 = B - B.mean(axis=0, keepdims=True)
        return (A0.T @ B0) / (A.shape[0] - 1)

    # ===== X, Y ως πολυδιάστατα blocks (40×3) =====
    X = data[:, :, i]  # (n_trials, 3)
    Y = data[:, :, j]  # (n_trials, 3)

    # ===== Επιλογή top-k για Z από cca_matrix =====
    candidates = [z for z in range(n_rois) if z != i and z != j]
    if len(candidates) > 0 and k > 0:
        rel = [max(cca_matrix[i, z], cca_matrix[j, z]) for z in candidates]
        order = np.argsort(rel)
        take = min(k, len(candidates))
        Z_idx = [candidates[idx] for idx in order[-take:]]
    else:
        Z_idx = []

    # ===== Χτίσιμο Z block (40×3k) =====
    if len(Z_idx) > 0:
        Z = np.concatenate([data[:, :, z] for z in Z_idx], axis=1)  # (n_trials, 3k)
    else:
        Z = np.zeros((n_trials, 0))

    # ===== Residualization: αφαιρώ Z από X και Y (κρατάνε shape (40×3)) =====
    if Z.size == 0:
        X_res = zscore(X)
        Y_res = zscore(Y)
    else:
        Xs = zscore(X)                # (n_trials, 3)
        Ys = zscore(Y)                # (n_trials, 3)
        Zs = zscore(Z)                # (n_trials, 3k)

        ZTZ = Zs.T @ Zs               # (3k, 3k)
        # Ridge λύση beta = (Z'Z + λI)^{-1} Z'X
        beta_X = np.linalg.pinv(ZTZ + ridge*np.eye(ZTZ.shape[0])) @ (Zs.T @ Xs)  # (3k, 3)
        beta_Y = np.linalg.pinv(ZTZ + ridge*np.eye(ZTZ.shape[0])) @ (Zs.T @ Ys)  # (3k, 3)

        X_hat = Zs @ beta_X           # (n_trials, 3)
        Y_hat = Zs @ beta_Y           # (n_trials, 3)

        X_res = Xs - X_hat            # (n_trials, 3)
        Y_res = Ys - Y_hat            # (n_trials, 3)

    # ===== CCA στα υπολείμματα (χωρίς sklearn) =====
    # Συνδιακυμάνσεις
    Sxx = cov(X_res, X_res)
    Syy = cov(Y_res, Y_res)
    Sxy = cov(X_res, Y_res)
    Syx = Sxy.T

    # Ridge στα diagonals για σταθερότητα
    Sxx_r = Sxx + ridge * np.eye(Sxx.shape[0])
    Syy_r = Syy + ridge * np.eye(Syy.shape[0])

    # Γενικευμένο ιδιοπρόβλημα: eig( Sxx^{-1} Sxy Syy^{-1} Syx )
    try:
        Sxx_inv = np.linalg.pinv(Sxx_r)
        Syy_inv = np.linalg.pinv(Syy_r)
        M = Sxx_inv @ Sxy @ Syy_inv @ Syx
        eigvals = np.linalg.eigvals(M)
        eigvals = np.real(eigvals)
        eigvals = np.clip(eigvals, 0.0, 1.0)
        rho = float(np.sqrt(np.max(eigvals))) if eigvals.size > 0 else 0.0
        return float(np.clip(rho, 0.0, 1.0))
    except Exception:
        return 0.0


In [ ]:
import numpy as np

def compute_pcca_matrix_conditionwise(data, cca_matrix, k=8, ridge=1e-3):
    """
    CONDITION-WISE PCCA
    Υπολογίζει το πλήρες PCCA matrix για ΟΛΑ τα trials μιας συνθήκης.
    
    Parameters:
    - data: shape (n_trials, 3, n_rois)
    - cca_matrix: shape (n_rois, n_rois)  (condition-wise CCA matrix)
    - k: πλήθος ROIs στο Z
    - ridge: ridge στα residualizations / inversions (περνά στο callee)

    Returns:
    - W: (n_rois, n_rois) PCCA matrix
    """
    n_trials, _, n_rois = data.shape
    W = np.zeros((n_rois, n_rois), dtype=float)
    np.fill_diagonal(W, 1.0)

    for i in range(n_rois):
        for j in range(i + 1, n_rois):
            rho = compute_pcca_condition_wise_with_topkZ(
                data, i, j, cca_matrix, k=k, ridge=ridge  
            )
            W[i, j] = rho
            W[j, i] = rho
    return W


In [14]:
task_pcca_dict = {}
rest_pcca_dict = {}

for sid in subject_ids:
    # subject_task_data[sid] : (n_trials, 3, n_rois)
    # task_cca_dict[sid]     : (n_rois, n_rois)
    W_task = compute_pcca_matrix_conditionwise(
        subject_task_data[sid],
        task_cca_dict[sid],
        k=2,
        ridge=1e-3
    )
    W_rest = compute_pcca_matrix_conditionwise(
        subject_rest_data[sid],
        rest_cca_dict[sid],
        k=8,
        ridge=1e-3
    )

    task_pcca_dict[sid] = W_task
    rest_pcca_dict[sid] = W_rest

    print(f"Subject {sid} task PCCA matrix: {W_task.shape}")
    print(f"Subject {sid} rest PCCA matrix: {W_rest.shape}")


Subject sub-0001 task PCCA matrix: (100, 100)
Subject sub-0001 rest PCCA matrix: (100, 100)
Subject sub-0002 task PCCA matrix: (100, 100)
Subject sub-0002 rest PCCA matrix: (100, 100)
Subject sub-0003 task PCCA matrix: (100, 100)
Subject sub-0003 rest PCCA matrix: (100, 100)
Subject sub-0004 task PCCA matrix: (100, 100)
Subject sub-0004 rest PCCA matrix: (100, 100)
Subject sub-0005 task PCCA matrix: (100, 100)
Subject sub-0005 rest PCCA matrix: (100, 100)
Subject sub-0006 task PCCA matrix: (100, 100)
Subject sub-0006 rest PCCA matrix: (100, 100)
Subject sub-0007 task PCCA matrix: (100, 100)
Subject sub-0007 rest PCCA matrix: (100, 100)
Subject sub-0008 task PCCA matrix: (100, 100)
Subject sub-0008 rest PCCA matrix: (100, 100)
Subject sub-0009 task PCCA matrix: (100, 100)
Subject sub-0009 rest PCCA matrix: (100, 100)
Subject sub-0011 task PCCA matrix: (100, 100)
Subject sub-0011 rest PCCA matrix: (100, 100)


In [54]:
from scipy.linalg import eig
import numpy as np

def compute_pcca_per_trial_with_topkZ(trial_data, i, j, cca_matrix, k=2):
    """
    Computes Partial Canonical Correlation (PCCA) between ROI i and j
    in a single trial, using top-k confounding ROIs selected from cca_matrix.
    
    Parameters:
    - trial_data: np.array of shape (3, n_rois)
    - i, j: indices of X and Y
    - cca_matrix: np.array of shape (n_rois, n_rois), CCA values between ROIs
    - k: number of top Z ROIs to use for conditioning
    
    Returns:
    - rho: partial canonical correlation (scalar between 0 and 1)
    """
    n_rois = trial_data.shape[1]

    # === Extract X and Y ===
    X = trial_data[:, i].reshape(-1, 1)
    Y = trial_data[:, j].reshape(-1, 1)

    # === Select top-k Z ROIs based on relevance in cca_matrix ===
    candidate_indices = [z for z in range(n_rois) if z != i and z != j]
    relevance_scores = [max(cca_matrix[i, z], cca_matrix[j, z]) for z in candidate_indices]
    top_k_indices = [candidate_indices[z] for z in np.argsort(relevance_scores)[-k:]]
    Z = trial_data[:, top_k_indices]  # shape (3, k)

    def center(A): return A - A.mean(axis=0, keepdims=True)
    def cov(A, B): return center(A).T @ center(B) / (A.shape[0] - 1)

    try:
        # === Compute covariances ===
        Sxz = cov(X, Z)
        Syz = cov(Y, Z)
        Szz = cov(Z, Z)
        Sxy = cov(X, Y)
        Sxx = cov(X, X)
        Syy = cov(Y, Y)

        # === Compute inverse safely ===
        inv_Szz = np.linalg.pinv(Szz)

        # === Partial covariances ===
        Sxy_z = Sxy - Sxz @ inv_Szz @ Syz.T
        Syx_z = Sxy_z.T
        Sxx_z = Sxx - Sxz @ inv_Szz @ Sxz.T
        Syy_z = Syy - Syz @ inv_Szz @ Syz.T

        # === Check trace values before sqrt ===
        trace_x = np.trace(Sxx_z)
        trace_y = np.trace(Syy_z)

        if not np.isfinite(trace_x) or not np.isfinite(trace_y):
            return 0.0
        if trace_x <= 0 or trace_y <= 0:
            return 0.0

        norm_x = np.sqrt(trace_x)
        norm_y = np.sqrt(trace_y)

        # === Normalize ===
        Sxy_z_norm = Sxy_z / (norm_x * norm_y)
        Syx_z_norm = Sxy_z_norm.T

        A = np.block([
            [np.zeros_like(Sxy_z_norm), Sxy_z_norm],
            [Syx_z_norm, np.zeros_like(Syx_z_norm)]
        ])
        B = np.block([
            [np.eye(Sxx_z.shape[0]), np.zeros_like(Sxy_z_norm)],
            [np.zeros_like(Syx_z_norm), np.eye(Syy_z.shape[0])]
        ])

        eigvals, _ = eig(A, B)
        eigvals = np.real(eigvals)
        canonical_corrs = np.sort(np.abs(eigvals))[::-1]

        rho = canonical_corrs[0] if canonical_corrs.size > 0 else 0.0
        return np.clip(rho, 0, 1)

    except Exception:
        return 0.0

In [55]:
def compute_pcca_tensor(data, cca_matrix, k=2):
    """
    Computes the full PCCA tensor:
    shape = (n_trials, n_rois, n_rois)

    Parameters:
    - data: shape (n_trials, 3, n_rois)
    - cca_matrix: shape (n_rois, n_rois), from compute_cca_matrix_across_trials
    - k: number of top confounding ROIs to use in Z

    Returns:
    - pcca_tensor: (n_trials, n_rois, n_rois)
    """
    n_trials, _, n_rois = data.shape
    pcca_tensor = np.zeros((n_trials, n_rois, n_rois))

    for t in range(n_trials):
        trial = data[t]  # shape (3, n_rois)
        for i in range(n_rois):
            for j in range(i + 1, n_rois):
                pcca_val = compute_pcca_per_trial_with_topkZ(trial, i, j, cca_matrix, k)
                pcca_tensor[t, i, j] = pcca_val
                pcca_tensor[t, j, i] = pcca_val  

    return pcca_tensor

In [56]:
task_pcca_dict = {}
rest_pcca_dict = {}

for sid in subject_ids:
    task_pcca = compute_pcca_tensor(subject_task_data[sid],task_cca_dict[sid])
    rest_pcca = compute_pcca_tensor(subject_rest_data[sid],rest_cca_dict[sid])

    task_pcca_dict[sid]=task_pcca
    rest_pcca_dict[sid]=rest_pcca
    print(f"Subject {sid} task data pcca",task_pcca_dict[sid].shape)
    print(f"Subject {sid} rest data pcca",rest_pcca_dict[sid].shape)

Subject sub-0001 task data pcca (40, 100, 100)
Subject sub-0001 rest data pcca (40, 100, 100)
Subject sub-0002 task data pcca (40, 100, 100)
Subject sub-0002 rest data pcca (40, 100, 100)
Subject sub-0003 task data pcca (40, 100, 100)
Subject sub-0003 rest data pcca (40, 100, 100)
Subject sub-0004 task data pcca (40, 100, 100)
Subject sub-0004 rest data pcca (40, 100, 100)
Subject sub-0005 task data pcca (40, 100, 100)
Subject sub-0005 rest data pcca (40, 100, 100)
Subject sub-0006 task data pcca (40, 100, 100)
Subject sub-0006 rest data pcca (40, 100, 100)
Subject sub-0007 task data pcca (40, 100, 100)
Subject sub-0007 rest data pcca (40, 100, 100)
Subject sub-0008 task data pcca (40, 100, 100)
Subject sub-0008 rest data pcca (40, 100, 100)
Subject sub-0009 task data pcca (40, 100, 100)
Subject sub-0009 rest data pcca (40, 100, 100)
Subject sub-0011 task data pcca (40, 100, 100)
Subject sub-0011 rest data pcca (40, 100, 100)


In [21]:
import os
import pandas as pd

def save_all_correlations_to_csv(subject_ids, output_dir="correlation_csvs"):
    os.makedirs(output_dir, exist_ok=True)

    for sid in subject_ids:
        print(f"Saving correlations for {sid}...")

        matrices = {
            "task_pearson": task_corr_dict[sid],
            "rest_pearson": rest_corr_dict[sid],
            # "task_partial": task_part_corr_dict[sid],
            # "rest_partial": rest_part_corr_dict[sid],
            "task_cca": task_cca_dict[sid],
            "rest_cca": rest_cca_dict[sid],
            "task_pcca": task_pcca_dict[sid],   # 👈 πρόσθεσε αυτά
            "rest_pcca": rest_pcca_dict[sid]  
        }

        for name, matrix in matrices.items():
            df = pd.DataFrame(matrix)
            df.to_csv(os.path.join(output_dir, f"{sid}_{name}.csv"), index=False)

    print(f"All CSVs saved in: {os.path.abspath(output_dir)}")


save_all_correlations_to_csv(subject_ids)

Saving correlations for sub-0001...
Saving correlations for sub-0002...
Saving correlations for sub-0003...
Saving correlations for sub-0004...
Saving correlations for sub-0005...
Saving correlations for sub-0006...
Saving correlations for sub-0007...
Saving correlations for sub-0008...
Saving correlations for sub-0009...
Saving correlations for sub-0011...
All CSVs saved in: c:\Users\elena\Documents\Thesis\Elena Kalla\preprocessed data\correlation_csvs


In [19]:
import os
import numpy as np
import pandas as pd

def save_pcca_matrix(W, sid, condition, output_dir="preprocessed data/correlation_csvs"):
    """
    Αποθηκεύει την PCCA μήτρα W (N×N) ως CSV ΧΩΡΙΣ header/index.
    Το format είναι ίδιο με των άλλων (pearson/partial/cca).
    """
    os.makedirs(output_dir, exist_ok=True)
    W = np.array(W, dtype=float)

    # προαιρετική εξομάλυνση/ασφάλεια
    W = 0.5 * (W + W.T)          # συμμετρία
    np.fill_diagonal(W, 1.0)     # διαγώνιος = 1
    W = np.nan_to_num(W, nan=0.) # καθάρισμα NaN

    out_path = os.path.join(output_dir, f"{sid}_{condition}_pcca.csv")
    pd.DataFrame(W).to_csv(out_path, header=False, index=False)
    print(f"✅ Saved PCCA matrix: {out_path}")

In [20]:
for sid in subject_ids:
    # αποθήκευση ως 100×100 matrix, ίδιο format με τα άλλα
    save_pcca_matrix(W_task, sid, "task", output_dir="preprocessed data/correlation_csvs")
    save_pcca_matrix(W_rest, sid, "rest", output_dir="preprocessed data/correlation_csvs")

✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0001_task_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0001_rest_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0002_task_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0002_rest_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0003_task_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0003_rest_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0004_task_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0004_rest_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0005_task_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0005_rest_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0006_task_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_csvs\sub-0006_rest_pcca.csv
✅ Saved PCCA matrix: preprocessed data/correlation_c

In [58]:
import pandas as pd
import os

def save_pcca_3d_to_csv(pcca_tensor, sid, condition, output_dir="correlation_csvs"):
    """
    Saves a (n_trials, n_rois, n_rois) PCCA tensor to a long-format CSV file.

    Parameters:
        pcca_tensor: np.array (n_trials, n_rois, n_rois)
        sid: subject ID (e.g. "sub-0001")
        condition: "task" or "rest"
        output_dir: folder to save the CSV
    """
    os.makedirs(output_dir, exist_ok=True)
    n_trials, n_rois, _ = pcca_tensor.shape

    records = []
    for t in range(n_trials):
        for i in range(n_rois):
            for j in range(n_rois):
                records.append({
                    "trial": t,
                    "roi_i": i,
                    "roi_j": j,
                    "pcca_value": pcca_tensor[t, i, j]
                })

    df = pd.DataFrame(records)
    df.to_csv(f"{output_dir}/{sid}_{condition}_pcca_3d.csv", index=False)
    print(f"✅ Saved: {output_dir}/{sid}_{condition}_pcca_3d.csv")
# %%
for sid in subject_ids:
    save_pcca_3d_to_csv(task_pcca_dict[sid], sid, "task")
    save_pcca_3d_to_csv(rest_pcca_dict[sid], sid, "rest")


✅ Saved: correlation_csvs/sub-0001_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0001_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0002_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0002_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0003_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0003_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0004_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0004_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0005_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0005_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0006_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0006_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0007_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0007_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0008_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0008_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0009_task_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0009_rest_pcca_3d.csv
✅ Saved: correlation_csvs/sub-0011_task_pcca_3d.csv
✅ Saved: cor